# Auto MPG — Regression Benchmark (Dataset #1)

**Domain:** Automotive | **Task:** Predict fuel efficiency (MPG) from engine specs

This notebook follows the curriculum workflow:
**baseline → primary → one advanced → single comparison table → resume bullet.**

- **Baseline:** mean predictor (DummyRegressor)
- **Primary:** Linear / Ridge / Lasso regression
- **Advanced:** Polynomial features (deg 2–3) + SVR
- **Metrics:** MAE, RMSE, R² (5-fold CV on train, final hold-out test)

> Data source: UCI Machine Learning Repository `auto-mpg` (identical content to the
> Kaggle `uciml/autompg-dataset`). Loaded from the canonical UCI URL for reproducibility
> without Kaggle API keys.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:
# Load Auto MPG (UCI canonical source; '?' marks missing horsepower)
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
COLS = ["mpg","cylinders","displacement","horsepower","weight",
        "acceleration","model_year","origin","car_name"]
df = pd.read_csv(URL, sep=r"\s+", header=None, names=COLS, na_values="?")

import os
os.makedirs("data", exist_ok=True)
df.to_csv("data/auto-mpg.csv", index=False)  # save raw copy for the Streamlit app

print(f"Rows: {len(df)}, Cols: {df.shape[1]}")
df.head()

Rows: 398, Cols: 9


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name
0,18.0,8,307.0,130.0,3504.0,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693.0,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150.0,3436.0,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150.0,3433.0,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140.0,3449.0,10.5,70,1,ford torino


## 1. EDA & Preprocessing

In [3]:
# Missing values: only 'horsepower' has holes
print("Missing per column:")
print(df.isna().sum()[lambda s: s > 0])

Missing per column:
horsepower    6
dtype: int64


In [4]:
# Numeric summary of the target and features
df[["mpg","cylinders","displacement","horsepower","weight","acceleration"]].describe().round(2)

,mpg,cylinders,displacement,horsepower,weight,acceleration
count,398.00,398.00,398.00,392.00,398.00,398.00
mean,23.51,5.45,193.43,104.47,2970.42,15.57
std,7.82,1.70,104.27,38.49,846.84,2.76
min,9.00,3.00,68.00,46.00,1613.00,8.00
25%,17.50,4.00,104.25,75.00,2223.75,13.82
50%,23.00,4.00,148.50,93.50,2803.50,15.50
75%,29.00,8.00,262.00,126.00,3608.00,17.18
max,46.60,8.00,455.00,230.00,5140.00,24.80


In [5]:
# Impute missing horsepower with the median, drop the car-name string column
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())
df = df.drop(columns=["car_name"])
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,18.0,8,307.0,130.0,3504.0,12.0,70,1
1,15.0,8,350.0,165.0,3693.0,11.5,70,1
2,18.0,8,318.0,150.0,3436.0,11.0,70,1
3,16.0,8,304.0,150.0,3433.0,12.0,70,1
4,17.0,8,302.0,140.0,3449.0,10.5,70,1


## 2. Train / Test Split + Baseline

In [6]:
feature_cols = ["cylinders","displacement","horsepower","weight",
                 "acceleration","model_year","origin"]
X = df[feature_cols]
y = df["mpg"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

Train: 318  |  Test: 80


In [7]:
# Baseline: predict the training-set mean (DummyRegressor)
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
y_base = baseline.predict(X_test)
print(f"Baseline  MAE={mean_absolute_error(y_test, y_base):.3f}  "
      f"RMSE={np.sqrt(mean_squared_error(y_test, y_base)):.3f}  "
      f"R2={r2_score(y_test, y_base):.3f}")

Baseline  MAE=5.955  RMSE=7.347  R2=-0.004


## 3. Primary Methods — Linear / Ridge / Lasso

In [8]:
def make_model(name, est, scale=False):
    steps = []
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", est))
    return name, Pipeline(steps)

primary = [
    make_model("LinearRegression", LinearRegression()),
    make_model("Ridge (alpha=1.0)", Ridge(alpha=1.0), scale=True),
    make_model("Lasso (alpha=0.01)", Lasso(alpha=0.01, max_iter=10000), scale=True),
]

cv_rows = []
for name, pipe in primary:
    cv_rmse = -cross_val_score(pipe, X_train, y_train, cv=5,
                               scoring="neg_root_mean_squared_error").mean()
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    cv_rows.append({
        "model": name,
        "cv_rmse": cv_rmse,
        "test_mae": mean_absolute_error(y_test, y_pred),
        "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "test_r2": r2_score(y_test, y_pred),
    })
    print(f"{name:22s} CV-RMSE={cv_rmse:.3f}  test RMSE={cv_rows[-1]['test_rmse']:.3f}  R2={cv_rows[-1]['test_r2']:.3f}")

LinearRegression       CV-RMSE=3.497  test RMSE=2.863  R2=0.848


Ridge (alpha=1.0)      CV-RMSE=3.493  test RMSE=2.865  R2=0.847


Lasso (alpha=0.01)     CV-RMSE=3.495  test RMSE=2.881  R2=0.846


## 4. Advanced — Polynomial Features + SVR

In [9]:
advanced = [
    make_model("Poly(deg2)+Ridge",
               Pipeline([("poly", PolynomialFeatures(degree=2, include_bias=False)),
                         ("ridge", Ridge(alpha=1.0))]), scale=True),
    make_model("Poly(deg3)+Ridge",
               Pipeline([("poly", PolynomialFeatures(degree=3, include_bias=False)),
                         ("ridge", Ridge(alpha=1.0))]), scale=True),
    make_model("SVR (RBF)",
               SVR(kernel="rbf", C=10.0, epsilon=0.1), scale=True),
]

for name, pipe in advanced:
    cv_rmse = -cross_val_score(pipe, X_train, y_train, cv=5,
                               scoring="neg_root_mean_squared_error").mean()
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    cv_rows.append({
        "model": name,
        "cv_rmse": cv_rmse,
        "test_mae": mean_absolute_error(y_test, y_pred),
        "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "test_r2": r2_score(y_test, y_pred),
    })
    print(f"{name:22s} CV-RMSE={cv_rmse:.3f}  test RMSE={cv_rows[-1]['test_rmse']:.3f}  R2={cv_rows[-1]['test_r2']:.3f}")

Poly(deg2)+Ridge       CV-RMSE=3.017  test RMSE=2.381  R2=0.895
Poly(deg3)+Ridge       CV-RMSE=3.652  test RMSE=2.412  R2=0.892


SVR (RBF)              CV-RMSE=2.975  test RMSE=2.088  R2=0.919


## 5. Comparison Table (all methods)

In [10]:
results = pd.DataFrame(cv_rows).sort_values("test_rmse").reset_index(drop=True)
results.insert(0, "rank", results.index + 1)
results.round(3)

# Save comparison table for the Streamlit dashboard
import os
os.makedirs("models", exist_ok=True)
results.to_csv("models/comparison.csv", index=False)
results.round(3)

,rank,model,cv_rmse,test_mae,test_rmse,test_r2
0,1,SVR (RBF),2.975,1.693,2.088,0.919
1,2,Poly(deg2)+Ridge,3.017,1.804,2.381,0.895
2,3,Poly(deg3)+Ridge,3.652,1.752,2.412,0.892
3,4,LinearRegression,3.497,2.255,2.863,0.848
4,5,Ridge (alpha=1.0),3.493,2.251,2.865,0.847
5,6,Lasso (alpha=0.01),3.495,2.255,2.881,0.846


In [11]:
best = results.iloc[0]
print("BEST MODEL:", best["model"])
print(f"  CV-RMSE = {best['cv_rmse']:.3f} | Test MAE = {best['test_mae']:.3f} | "
      f"Test RMSE = {best['test_rmse']:.3f} | Test R2 = {best['test_r2']:.3f}")

BEST MODEL: SVR (RBF)
  CV-RMSE = 2.975 | Test MAE = 1.693 | Test RMSE = 2.088 | Test R2 = 0.919


## 6. Export Best Model (for Streamlit deployment)

In [12]:
import joblib, json, os

# Rebuild the winning pipeline definition and refit on ALL data for deployment
all_pipes = primary + advanced
best_pipe = next(p for n, p in all_pipes if n == best["model"])
best_pipe.fit(X, y)  # refit on full dataset (metrics above stay hold-out based)

os.makedirs("models", exist_ok=True)
joblib.dump(best_pipe, "models/best_model.joblib")

meta = {
    "dataset": "Auto MPG",
    "target": "mpg",
    "features": feature_cols,
    "best_model": best["model"],
    "metrics": {
        "cv_rmse": round(float(best["cv_rmse"]), 3),
        "test_mae": round(float(best["test_mae"]), 3),
        "test_rmse": round(float(best["test_rmse"]), 3),
        "test_r2": round(float(best["test_r2"]), 3),
    },
    "n_samples": int(len(df)),
    "loader": "joblib.load('models/best_model.joblib')",
}
with open("models/model_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved models/best_model.joblib + models/model_metadata.json")
print(json.dumps(meta, indent=2))

Saved models/best_model.joblib + models/model_metadata.json
{
  "dataset": "Auto MPG",
  "target": "mpg",
  "features": [
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "acceleration",
    "model_year",
    "origin"
  ],
  "best_model": "SVR (RBF)",
  "metrics": {
    "cv_rmse": 2.975,
    "test_mae": 1.693,
    "test_rmse": 2.088,
    "test_r2": 0.919
  },
  "n_samples": 398,
  "loader": "joblib.load('models/best_model.joblib')"
}


## 7. Resume Bullet

In [13]:
delta = best["test_rmse"] - float(results[results['model'] == 'LinearRegression']['test_rmse'].iloc[0])
bullet = (
    f"Benchmarked {len(results)} regression methods on Auto MPG ({len(df)} cars, "
    f"{len(feature_cols)} features); achieved {best['model']} at test RMSE "
    f"{best['test_rmse']:.2f} MPG (R2 {best['test_r2']:.3f}, MAE {best['test_mae']:.2f} mpg) "
    f"- {abs(delta):.2f} RMSE vs baseline linear."
)
print("Resume bullet (Auto MPG):")
print("\u2022 " + bullet)

Resume bullet (Auto MPG):
• Benchmarked 6 regression methods on Auto MPG (398 cars, 7 features); achieved SVR (RBF) at test RMSE 2.09 MPG (R2 0.919, MAE 1.69 mpg) - 0.78 RMSE vs baseline linear.
